## I2 reader mode: Small CONSTRUCTED execution + current accepted historical summary

All newly computed values are **CONSTRUCTED method demonstrations**, never market observations or historical verification. Historical tables are separately labelled replay. French raw/extracted/processed bytes are not bundled. French route: A10-A limited-scope design accepted, execution blocked; A10-A.1 protocol technically accepted pending permissions and execution authorization. G01 rights/subject/use unknown; G05 A10-B not authorized.

Original explanatory prose below describes the course method. The reader-mode code and labels define which portions execute here. No P6 retraining or Ch39 historical row reconstruction.


# Project 4 Cost & Execution Lab: Interactive Teaching Notebook

**Module**: Project 4 Multi-Factor Strategy Execution Lab  
**Contract**: `qa/a7/ch35_ch36_batch/INTERFACE_CONTRACT.md`  
**Data Status**: CONSTRUCTED TEACHING DATA (Seed `20261210`; Not Empirical)

This notebook provides an interactive, reproducible environment to explore:
1. Slicing algorithms: **TWAP**, **forecast VWAP**, and **lagged-observed POV**.
2. **Partial fills**, cutoff remainder ($Q = F + R$), and zero-fill boundary behavior.
3. **Implementation Shortfall (IS)** direct identity vs 4-part decomposition.
4. **Balance sheet conservation**: cash, positions, NAV, and event-to-event ACT/360 cash interest and borrow accounting. Each event settles the prior balance and mark before applying the newly observed fill or mark; accrued cash interest is posted to cash per event.
5. **Walk-forward evaluation**: 60-month train / 36-month test split with a frozen volume profile.
6. Compare gross and net outcomes under a clearly labeled constructed cost scenario.


In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("cost_execution_lab_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from quant_course.execution_costs import ExecutionConfig
from projects.project04.cost_execution_lab.p4_cost_bridge import estimate_frozen_volume_profile, run_fixed_parent_comparison, run_partial_and_zero_fill_demo
print("Small CONSTRUCTED execution examples + accepted 2026-10-14 current-signal summary replay; full P4 strategy SKIPPED")


## Step 1: Configuration & Frozen Intraday Volume Profile

The Notebook reads the same declared config and constructed-universe dimensions as the CLI. Per `INTERFACE_CONTRACT.md`, the intraday volume forecast curve used by VWAP is frozen before testing. Here we estimate a U-shaped volume curve across the trading day (09:30 to 16:00 UTC).


In [ ]:
CONFIG_PATH = ROOT / "projects/project04/cost_execution_lab/config.json"
with CONFIG_PATH.open(encoding="utf-8") as stream:
    config_data = json.load(stream)
authoritative = json.loads((ROOT / "fixtures/projects/project04/summary_results_current_signal_20261014.json").read_text(encoding="utf-8"))
SEED = config_data["universe"]["seed"]
frozen_profile = estimate_frozen_volume_profile(
    n_slots=config_data["intraday_bridge"]["slots_per_day"], seed=SEED
)

plt.figure(figsize=(8, 4))
slots_x = [f"Slot {i}\n({9 + (30*i)//60:02d}:{(30*i)%60:02d})" for i in range(13)]
plt.plot(range(13), frozen_profile * 100, marker='o', color='#2b83ba', linewidth=2)
plt.title("Train-Only Frozen Intraday Volume Profile (U-Shape)", fontweight="bold")
plt.xlabel("Intraday 30-Minute Slot")
plt.ylabel("Expected Volume Share (%)")
plt.xticks(range(13), slots_x, rotation=45, ha='right', fontsize=8)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(RUN_DIR / "constructed_volume_profile.png", dpi=130)
plt.show()

print(f"Volume profile sums to: {frozen_profile.sum():.6f}")


## Step 2: Fixed Parent Order Slicing Comparison (TWAP vs VWAP vs POV)

We execute an identical benchmark parent order of **10,000 shares Buy** across 13 slots under identical market conditions.
We verify:
- $Q = F + R$
- Fill VWAP
- IS 4-part decomposition vs Direct Identity


In [ ]:
cost_cfg = config_data["execution_cost_model"]
cfg = ExecutionConfig(
    commission_rate=cost_cfg["commission_rate"],
    commission_per_share=cost_cfg["commission_per_share"],
    half_spread_bps=cost_cfg["half_spread_bps"],
    temp_impact_coef=cost_cfg["temp_impact_coef"],
    temp_impact_exp=cost_cfg["temp_impact_exp"],
    perm_impact_coef=cost_cfg["perm_impact_coef"],
    max_participation=cost_cfg["max_participation"],
    cash_rate_annual=cost_cfg["cash_rate_annual"],
    cash_debit_rate_annual=cost_cfg["cash_debit_rate_annual"],
    borrow_rate_annual=cost_cfg["borrow_rate_annual"],
    day_count_basis=cost_cfg["day_count_basis"],
)

fixed_res = run_fixed_parent_comparison(cfg, frozen_profile, seed=SEED)
for side, algorithms in fixed_res.items():
    for algorithm, row in algorithms.items():
        expected = authoritative["experiment_1_fixed_parent_comparison"][side][algorithm]
        for metric in ("total_is", "total_is_bps"):
            assert np.isclose(row[metric], expected[metric], rtol=1e-10, atol=1e-10)
        assert row["identity_verified"] == expected["identity_verified"]
buy_comp = fixed_res["BUY"]

df_comp = pd.DataFrame(buy_comp).T[[
    "parent_quantity", "filled_quantity", "fill_rate", "fill_vwap",
    "delay_cost", "realized_execution_cost", "explicit_fees", "total_is", "total_is_bps", "identity_verified"
]]
display(df_comp)


## Step 3: Partial Fills, Remainder Cutoff & Zero-Fill Handling

Under restrictive market liquidity, child orders are capped by participation limits. Unfilled shares become remainder $R = Q - F$.
At 0% participation:
- Filled shares $F = 0$
- **Fill VWAP is strictly `None`**
- Explicit commission is **$0.00**
- Opportunity cost equals $Q \cdot (P_{\text{terminal}} - P_{\text{arrival}})$


In [ ]:
stress_res = run_partial_and_zero_fill_demo(frozen_profile, seed=SEED)
for row, expected in zip(stress_res, authoritative["experiment_2_liquidity_stress"]):
    for metric in ("filled_quantity", "remainder_quantity", "realized_execution_cost", "unfilled_opportunity_cost", "explicit_fees", "total_is"):
        assert np.isclose(row[metric], expected[metric], rtol=1e-10, atol=1e-10)
    if row["fill_vwap"] is None or expected["fill_vwap"] is None:
        assert row["fill_vwap"] is expected["fill_vwap"]
    else:
        assert np.isclose(row["fill_vwap"], expected["fill_vwap"], rtol=1e-10, atol=1e-10)
df_stress = pd.DataFrame(stress_res)
display(df_stress[[
    "max_participation", "parent_quantity", "filled_quantity", "remainder_quantity",
    "fill_fraction", "fill_vwap", "realized_execution_cost", "unfilled_opportunity_cost", "total_is", "identity_verified"
]])


## Step 4: Chronological Walk-Forward Multi-Factor Backtest

We load the A5 signal layer (Value, Momentum, Quality composite), construct month-end target weights, and run the 96-month walk-forward backtest (60 months train, 36 months test). Returns use close-to-close month-end NAVs; sizing and turnover use the NAV after financing is settled at rebalance. The 13 half-hour execution slots describe one monthly rebalance window. Turnover includes only filled risky-security notional; cash-weight changes do not create stock fills or fees. Annual turnover is the mean monthly rate multiplied by 12.


In [ ]:
perf_dict = authoritative["experiment_3_walk_forward_performance"]
rows = [{"strategy": strategy, "window": window, **values} for strategy, windows in perf_dict.items() for window, values in windows.items()]
display(pd.DataFrame(rows))
print("Historical accepted CONSTRUCTED summary replay; no walk-forward run or new market verification")


## Step 5: Constructed High-Friction Scenario

This scenario compares gross and net results under specified synthetic costs. It illustrates one model path and does not establish whether real-market alpha survives costs.


In [ ]:
display(pd.Series(authoritative["experiment_4_zero_negative_result"]))
print("Accepted historical negative/zero-result scenarios retained; no rerun")


## Step 6: Cost, Fill-Cap & Target-Scale Sensitivity

The low participation caps are chosen to bind in this constructed volume path. The target-scale sweep changes both requested turnover and gross exposure, so it is a joint exposure/turnover scenario, not an isolated turnover effect.


In [ ]:
sens_res = authoritative["experiment_5_sensitivity_analysis"]
display(pd.DataFrame(sens_res["fee_and_participation"]))
display(pd.DataFrame(sens_res["target_weight_scale"]))


## Step 7: Stepwise Friction Ablation Study

We decompose the cumulative drag of each friction layer:
1. Frictionless Gross
2. + Bid-Ask Half Spread
3. + Temporary Market Impact
4. + Broker Commissions
5. + Short Borrow Fees & Cash Financing


In [ ]:
df_abl = pd.DataFrame(authoritative["experiment_6_ablation_study"])
display(df_abl)
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(df_abl["stage"].str.replace("_"," "), df_abl["cagr"]*100)
ax.set(title="Historical accepted CONSTRUCTED ablation summary",ylabel="CAGR (%)")
ax.tick_params(axis="x",rotation=20); fig.tight_layout()
fig.savefig(RUN_DIR / "historical_constructed_ablation.png",dpi=130); plt.show()


## Step 8: Multi-Period Balance Sheet & Cash Ledger Audit

We verify that cash, positions, trades, commissions, interest, and borrow fees reconcile to machine precision:
$$\text{Cash}_{\text{end}} = \text{Cash}_{\text{start}} - \sum \text{Trades} - \sum \text{Fees} + \text{Interest} - \text{Borrow}$$


In [ ]:
display(pd.DataFrame(authoritative["ledger_audits"]).T)
print("Historical accepted audits replayed; no new ledger or full P4 run")
